# Robust Simulation Procedure for Tiny Permutation p-values

This notebook runs a **replicated, budget-controlled simulation study** for:
- IID naive sampling from the permutation space
- MCMC-IS with flat-tail exponential tilt
- SAMC (paper-consistent implementation)

It includes:
1. exact-ground-truth scenarios
2. explicit tuning vs production separation
3. equal computational budgets across methods
4. efficiency metrics (with/without tuning cost)
5. method-specific diagnostics

Run order:
1. Run imports
2. Set configuration (`FAST_MODE=True` first)
3. Run all cells top-to-bottom


In [2]:
from __future__ import annotations

from dataclasses import dataclass
from math import comb
from pathlib import Path
import sys
import json
import time

import numpy as np
import matplotlib.pyplot as plt

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from perm_pval.core.problem import PermutationTestProblem
from perm_pval.stats.ranks import mann_whitney_u
from perm_pval.stats.two_sample import difference_in_means
from perm_pval.exact.rank_sum_dp import RankSumDPSolver
from perm_pval.exact.linear_statistic_dp import LinearStatisticDPSolver
from perm_pval.methods.random_sampling import run_random_sampling
from perm_pval.methods.mcmc_is import run_mcmc_is
from perm_pval.methods.samc import run_samc
from perm_pval.methods.beta_tuning import (
    estimate_scale_T,
    iid_pilot_statistics,
    init_beta_from_iid_pilot,
    make_short_chain_q_runner,
    tune_beta_to_target_q,
)

print(f"Using project root: {project_root}")


ModuleNotFoundError: No module named 'perm_pval'

## Configuration

- Use `FAST_MODE=True` for quick smoke runs.
- Use `FAST_MODE=False` for publication-scale runs.
- `BUDGETS` is the per-method budget (same budget for IID, MCMC-IS, SAMC).
- `N_REPEATS` controls replication for variance/RMSE comparisons.


In [ ]:
FAST_MODE = True

if FAST_MODE:
    BUDGETS = [10_000, 25_000, 50_000]
    N_REPEATS = 6
    MCMC_PILOT_SAMPLES = 4_000
    MCMC_TUNE_STEPS = 2_500
    MCMC_TUNE_BURN_IN = 400
    SAMC_LAMBDA_MIN_PILOT = 2_000
else:
    BUDGETS = [100_000, 250_000, 500_000, 1_000_000]
    N_REPEATS = 20
    MCMC_PILOT_SAMPLES = 20_000
    MCMC_TUNE_STEPS = 12_000
    MCMC_TUNE_BURN_IN = 2_000
    SAMC_LAMBDA_MIN_PILOT = 10_000

# Scenario selection
SELECTED_SCENARIOS = [
    "rank_linear_x",
    "linear_nonlinear_x",
    "binary_hypergeom",
]

BASE_SEED = 12345

# MCMC-IS production settings
MCMC_CHAINS = 2
MCMC_BURN_FRAC = 0.20
MCMC_THIN = 1
MCMC_ESTIMATE_VARIANCE = True
MCMC_OBM_BATCH_SIZE = None

# MCMC beta initialization/tuning settings
MCMC_SCALE_METHOD = "sd"  # "sd" or "mad"
MCMC_P0_DESIGN = 1e-8
MCMC_D_ALPHA = 0.25
MCMC_USE_TRUE_P0_FOR_Q_TARGET = True  # if exact p is available in simulation
MCMC_Q_TARGET_DESIGN = MCMC_P0_DESIGN ** MCMC_D_ALPHA
MCMC_BETA_MAX_INIT = 1e6
MCMC_TUNE_THIN = 2
MCMC_TUNE_BRACKET_FACTOR = 2.0
MCMC_TUNE_TOL_REL = 0.20
MCMC_TUNE_MAX_BRACKET = 12
MCMC_TUNE_MAX_BISECT = 12
MCMC_TUNE_REPLICATE = 1
MCMC_TUNE_REUSE_STATE = True
MCMC_BETA_OVERRIDE = None  # set float to force beta

# SAMC settings (paper-consistent)
SAMC_BURN_FRAC = 0.20
SAMC_N_BINS = 40
SAMC_T0 = 1000.0
SAMC_TRACE_EVERY = 200
SAMC_PROPOSAL_FRAC = 0.05
SAMC_CONVERGENCE_TOL = 20.0

# Plot controls
LARGEST_BUDGET_ONLY_DIAGNOSTICS = True

print(json.dumps({
    "FAST_MODE": FAST_MODE,
    "BUDGETS": BUDGETS,
    "N_REPEATS": N_REPEATS,
    "SELECTED_SCENARIOS": SELECTED_SCENARIOS,
    "MCMC_CHAINS": MCMC_CHAINS,
    "MCMC_Q_TARGET_DESIGN": MCMC_Q_TARGET_DESIGN,
    "MCMC_USE_TRUE_P0_FOR_Q_TARGET": MCMC_USE_TRUE_P0_FOR_Q_TARGET,
    "MCMC_PILOT_SAMPLES": MCMC_PILOT_SAMPLES,
    "MCMC_TUNE_STEPS": MCMC_TUNE_STEPS,
    "SAMC_N_BINS": SAMC_N_BINS,
    "SAMC_T0": SAMC_T0,
    "SAMC_LAMBDA_MIN_PILOT": SAMC_LAMBDA_MIN_PILOT,
}, indent=2))


## Scenario and Method Helpers

In [ ]:
@dataclass
class Scenario:
    name: str
    display_name: str
    problem: PermutationTestProblem
    exact_p: float
    exact_tail_hits: int
    exact_n_perm: int
    exact_method: str
    notes: str


def near_tail_labels_tiny_p_case() -> tuple[np.ndarray, np.ndarray]:
    n = 40
    x = np.arange(1, n + 1, dtype=float)
    treated_values = set([1, 16] + list(range(23, 41)))
    y = np.array([1 if int(v) in treated_values else 0 for v in x], dtype=np.int8)
    return x, y


def treated_successes(x: np.ndarray, y: np.ndarray) -> float:
    x_arr = np.asarray(x, dtype=np.int8)
    y_arr = np.asarray(y, dtype=np.int8)
    return float(int(np.dot(x_arr, y_arr)))


def exact_hypergeom_right_tail(n: int, n_treated: int, total_successes: int, k_obs: int) -> tuple[float, int, int]:
    total_states = comb(n, n_treated)
    k_max = min(total_successes, n_treated)
    k_min = max(0, n_treated - (n - total_successes))
    tail_hits = 0
    for k in range(max(k_obs, k_min), k_max + 1):
        tail_hits += comb(total_successes, k) * comb(n - total_successes, n_treated - k)
    return float(tail_hits / total_states), int(tail_hits), int(total_states)


def build_scenarios() -> dict[str, Scenario]:
    scenarios: dict[str, Scenario] = {}

    # Scenario 1: rank-sum U with x=1..40
    x, y = near_tail_labels_tiny_p_case()
    p_rank = PermutationTestProblem(X=x, y_obs=y, statistic=mann_whitney_u, tail="right")
    ex_rank = RankSumDPSolver(p_rank, statistic_type="u").compute()
    scenarios["rank_linear_x"] = Scenario(
        name="rank_linear_x",
        display_name="Rank-sum U on x=1..40",
        problem=p_rank,
        exact_p=ex_rank.p_value,
        exact_tail_hits=ex_rank.tail_hits,
        exact_n_perm=ex_rank.n_permutations,
        exact_method="RankSumDPSolver",
        notes="Reference rank-based exact DP scenario.",
    )

    # Scenario 2: difference in means on nonlinear x=i^2
    x_nl = x**2
    p_lin = PermutationTestProblem(X=x_nl, y_obs=y, statistic=difference_in_means, tail="right")
    ex_lin = LinearStatisticDPSolver.from_difference_in_means(p_lin, score_scale=1).compute()
    scenarios["linear_nonlinear_x"] = Scenario(
        name="linear_nonlinear_x",
        display_name="Diff means on x=i^2",
        problem=p_lin,
        exact_p=ex_lin.p_value,
        exact_tail_hits=ex_lin.tail_hits,
        exact_n_perm=ex_lin.n_permutations,
        exact_method="LinearStatisticDPSolver",
        notes="Non-rank-linear numeric exact DP scenario.",
    )

    # Scenario 3: binary outcomes with hypergeometric exact tail
    n = 40
    n1 = 20
    m_success = 15
    y_bin = np.zeros(n, dtype=np.int8)
    y_bin[:n1] = 1
    x_bin = np.zeros(n, dtype=np.int8)
    x_bin[:m_success] = 1
    p_bin = PermutationTestProblem(X=x_bin, y_obs=y_bin, statistic=treated_successes, tail="right")
    p_hg, hg_hits, hg_total = exact_hypergeom_right_tail(
        n=n,
        n_treated=n1,
        total_successes=int(np.sum(x_bin)),
        k_obs=int(p_bin.t_obs),
    )
    # Optional DP cross-check (same linear statistic)
    ex_bin_dp = LinearStatisticDPSolver(
        p_bin,
        scores=x_bin.astype(float),
        score_scale=1,
        scale=1.0,
        offset=0.0,
    ).compute()
    if not np.isclose(p_hg, ex_bin_dp.p_value, atol=1e-15, rtol=0.0):
        raise RuntimeError("Hypergeometric exact p and DP exact p do not match.")

    scenarios["binary_hypergeom"] = Scenario(
        name="binary_hypergeom",
        display_name="Binary treated-successes",
        problem=p_bin,
        exact_p=p_hg,
        exact_tail_hits=hg_hits,
        exact_n_perm=hg_total,
        exact_method="Hypergeometric (cross-checked by Linear DP)",
        notes="Discrete null with exact hypergeometric tail.",
    )

    return scenarios


def tuning_eval_calls_from_history(history: list[dict]) -> int:
    eval_calls = 0
    for rec in history:
        stage = str(rec.get("stage", ""))
        if stage == "bisect":
            continue
        eval_calls += 1
    return eval_calls


def tune_mcmc_for_scenario(problem: PermutationTestProblem, exact_p: float, *, seed: int) -> dict:
    t0 = time.perf_counter()

    pilot_t = iid_pilot_statistics(problem, n_samples=MCMC_PILOT_SAMPLES, seed=seed)
    sigma_t = estimate_scale_T(pilot_t, method=MCMC_SCALE_METHOD)

    beta0_formula = float(np.sqrt(np.log(1.0 / exact_p)))
    p0_for_qtarget = float(exact_p) if MCMC_USE_TRUE_P0_FOR_Q_TARGET else float(MCMC_P0_DESIGN)
    q_target = float(p0_for_qtarget ** MCMC_D_ALPHA)
    beta0_laplace = init_beta_from_iid_pilot(
        pilot_T=pilot_t,
        T_obs=problem.t_obs,
        sigma_T=sigma_t,
        p0=p0_for_qtarget,
        q_target=q_target,
        beta_max=MCMC_BETA_MAX_INIT,
    )

    runner = make_short_chain_q_runner(problem, sigma_T=sigma_t, thin=MCMC_TUNE_THIN, seed=seed + 1)
    tuning = tune_beta_to_target_q(
        run_short_chain_fn=runner,
        init_state=problem.y_obs,
        beta0=beta0_laplace,
        q_target=q_target,
        n_steps=MCMC_TUNE_STEPS,
        burn_in=MCMC_TUNE_BURN_IN,
        bracket_factor=MCMC_TUNE_BRACKET_FACTOR,
        tol_rel=MCMC_TUNE_TOL_REL,
        max_bracket_iter=MCMC_TUNE_MAX_BRACKET,
        max_bisect_iter=MCMC_TUNE_MAX_BISECT,
        replicate=MCMC_TUNE_REPLICATE,
        reuse_state=MCMC_TUNE_REUSE_STATE,
    )

    beta_used = float(MCMC_BETA_OVERRIDE) if MCMC_BETA_OVERRIDE is not None else float(tuning["beta_hat"])

    eval_calls = tuning_eval_calls_from_history(tuning["history"])
    tuning_evals = int(MCMC_PILOT_SAMPLES + eval_calls * (MCMC_TUNE_STEPS + 1))
    tuning_time = float(time.perf_counter() - t0)

    return {
        "beta_formula": beta0_formula,
        "beta0_laplace": float(beta0_laplace),
        "beta_hat_tuned": float(tuning["beta_hat"]),
        "beta_used": beta_used,
        "sigma_t": float(sigma_t),
        "q_target": float(q_target),
        "p0_for_qtarget": float(p0_for_qtarget),
        "use_true_p0_for_qtarget": bool(MCMC_USE_TRUE_P0_FOR_Q_TARGET),
        "q_hat_beta_hat": float(tuning["q_hat"]),
        "bracket_succeeded": bool(tuning["bracket_succeeded"]),
        "history_tail": tuning["history"][-5:],
        "history_len": len(tuning["history"]),
        "pilot_min": float(np.min(pilot_t)),
        "pilot_max": float(np.max(pilot_t)),
        "tuning_eval_calls": int(eval_calls),
        "tuning_evals": tuning_evals,
        "tuning_time_sec": tuning_time,
    }


def tune_samc_for_scenario(problem: PermutationTestProblem, *, seed: int) -> dict:
    t0 = time.perf_counter()

    pilot_t = iid_pilot_statistics(problem, n_samples=SAMC_LAMBDA_MIN_PILOT, seed=seed)
    lambda_min = float(np.min(pilot_t))
    if not np.isfinite(lambda_min):
        lambda_min = float(problem.t_obs - 1.0)
    if lambda_min >= problem.t_obs:
        lambda_min = float(problem.t_obs - 1.0)

    finite_edges = np.linspace(lambda_min, float(problem.t_obs), SAMC_N_BINS, dtype=float)
    bin_edges = np.concatenate([finite_edges, np.asarray([np.inf], dtype=float)])

    tuning_time = float(time.perf_counter() - t0)
    return {
        "lambda_min": lambda_min,
        "bin_edges": bin_edges,
        "tuning_evals": int(SAMC_LAMBDA_MIN_PILOT),
        "tuning_time_sec": tuning_time,
        "pilot_min": float(np.min(pilot_t)),
        "pilot_max": float(np.max(pilot_t)),
    }


def run_iid_once(problem: PermutationTestProblem, *, budget: int, seed: int) -> dict:
    t0 = time.perf_counter()
    res = run_random_sampling(problem, n_samples=int(budget), seed=seed)
    dt = float(time.perf_counter() - t0)
    return {
        "method": "iid",
        "estimate": float(res.estimate),
        "variance_estimate": float(res.standard_error**2),
        "eval_excl_tuning": int(budget),
        "wall_time_sec": dt,
        "tail_hits": int(res.tail_hits),
        "tail_share_raw": float(res.estimate),
        "zero_hits": int(res.tail_hits == 0),
        "ess": np.nan,
        "acceptance_rate": np.nan,
        "q_tilt_tail_share": np.nan,
        "snis_mcse": np.nan,
        "samc_max_rel_freq_error": np.nan,
        "samc_converged": np.nan,
        "samc_pi0": np.nan,
    }


def run_mcmc_once(problem: PermutationTestProblem, *, budget: int, seed: int, tuning: dict) -> dict:
    steps_per_chain = max(int(budget) // MCMC_CHAINS, 1)
    total_steps = steps_per_chain * MCMC_CHAINS
    burn_in = min(int(MCMC_BURN_FRAC * steps_per_chain), max(steps_per_chain - 1, 0))

    t0 = time.perf_counter()
    res = run_mcmc_is(
        problem,
        beta=tuning["beta_used"],
        sigma_t=tuning["sigma_t"],
        n_steps=steps_per_chain,
        burn_in=burn_in,
        thin=MCMC_THIN,
        n_chains=MCMC_CHAINS,
        seed=seed,
        init="random",
        estimate_variance=MCMC_ESTIMATE_VARIANCE,
        obm_batch_size=MCMC_OBM_BATCH_SIZE,
    )
    dt = float(time.perf_counter() - t0)

    var_est = res.snis_variance_obm if (res.snis_variance_obm is not None and np.isfinite(res.snis_variance_obm)) else np.nan
    mcse = res.snis_mcse_obm if (res.snis_mcse_obm is not None and np.isfinite(res.snis_mcse_obm)) else np.nan

    return {
        "method": "mcmc_is",
        "estimate": float(res.estimate),
        "variance_estimate": float(var_est) if np.isfinite(var_est) else np.nan,
        "eval_excl_tuning": int(MCMC_CHAINS * (steps_per_chain + 1)),
        "wall_time_sec": dt,
        "tail_hits": int(res.tail_hits_weighted_sample),
        "tail_share_raw": float(res.tail_share_raw_sample),
        "zero_hits": int(res.tail_hits_weighted_sample == 0),
        "ess": float(res.ess),
        "acceptance_rate": float(res.overall_acceptance_rate),
        "q_tilt_tail_share": float(res.tail_share_raw_sample),
        "snis_mcse": float(mcse) if np.isfinite(mcse) else np.nan,
        "samc_max_rel_freq_error": np.nan,
        "samc_converged": np.nan,
        "samc_pi0": np.nan,
        "beta": float(tuning["beta_used"]),
        "sigma_t": float(tuning["sigma_t"]),
        "mcmc_total_steps": int(total_steps),
    }


def samc_variance_proxy(p_hat: float, n_steps: int, burn_in: int) -> float:
    n_eff = max(int(n_steps - burn_in), 1)
    return float(max(p_hat * (1.0 - p_hat) / n_eff, 0.0))


def run_samc_once(problem: PermutationTestProblem, *, budget: int, seed: int, tuning: dict) -> dict:
    n_steps = int(budget)
    burn_in = min(int(SAMC_BURN_FRAC * n_steps), max(n_steps - 1, 0))

    t0 = time.perf_counter()
    res = run_samc(
        problem,
        n_steps=n_steps,
        burn_in=burn_in,
        bin_edges=tuning["bin_edges"],
        seed=seed,
        init="random",
        t0=SAMC_T0,
        trace_every=SAMC_TRACE_EVERY,
        proposal_size=SAMC_PROPOSAL_FRAC,
        convergence_tolerance=SAMC_CONVERGENCE_TOL,
    )
    dt = float(time.perf_counter() - t0)

    return {
        "method": "samc",
        "estimate": float(res.estimate),
        "variance_estimate": samc_variance_proxy(float(res.estimate), n_steps=n_steps, burn_in=burn_in),
        "eval_excl_tuning": int(n_steps + 1),
        "wall_time_sec": dt,
        "tail_hits": np.nan,
        "tail_share_raw": np.nan,
        "zero_hits": np.nan,
        "ess": np.nan,
        "acceptance_rate": float(res.acceptance_rate),
        "q_tilt_tail_share": np.nan,
        "snis_mcse": np.nan,
        "samc_max_rel_freq_error": float(res.max_abs_relative_frequency_error),
        "samc_converged": int(res.convergence_reached),
        "samc_pi0": float(res.pi0_adjustment),
        "samc_estimator": str(res.pvalue_estimator),
    }


def run_single_budget_rep(
    scenario: Scenario,
    *,
    budget: int,
    rep: int,
    scenario_index: int,
    mcmc_tuning: dict,
    samc_tuning: dict,
) -> list[dict]:
    base = int(BASE_SEED + 1_000_000 * scenario_index + 10_000 * rep + budget)

    iid_row = run_iid_once(scenario.problem, budget=budget, seed=base + 1)
    mcmc_row = run_mcmc_once(scenario.problem, budget=budget, seed=base + 2, tuning=mcmc_tuning)
    samc_row = run_samc_once(scenario.problem, budget=budget, seed=base + 3, tuning=samc_tuning)

    tuning_eval = {
        "iid": 0,
        "mcmc_is": int(mcmc_tuning["tuning_evals"]),
        "samc": int(samc_tuning["tuning_evals"]),
    }
    tuning_time = {
        "iid": 0.0,
        "mcmc_is": float(mcmc_tuning["tuning_time_sec"]),
        "samc": float(samc_tuning["tuning_time_sec"]),
    }

    out = []
    for row in [iid_row, mcmc_row, samc_row]:
        est = float(row["estimate"])
        exact_p = float(scenario.exact_p)
        abs_log10_error = float(abs(np.log10(max(est, 1e-300)) - np.log10(exact_p)))
        out.append({
            "scenario": scenario.name,
            "scenario_display": scenario.display_name,
            "exact_p": exact_p,
            "budget": int(budget),
            "replicate": int(rep),
            **row,
            "bias": float(est - exact_p),
            "squared_error": float((est - exact_p) ** 2),
            "abs_log10_error": abs_log10_error,
            "eval_incl_tuning": int(row["eval_excl_tuning"] + tuning_eval[row["method"]]),
            "wall_time_incl_tuning_sec": float(row["wall_time_sec"] + tuning_time[row["method"]]),
        })
    return out


def run_scenario_experiment(scenario: Scenario, *, scenario_index: int, verbose: bool = True) -> tuple[list[dict], dict]:
    if verbose:
        print(f"\n=== Scenario: {scenario.display_name} ===")
        print(f"Exact p = {scenario.exact_p:.3e} ({scenario.exact_method})")

    mcmc_tuning = tune_mcmc_for_scenario(scenario.problem, scenario.exact_p, seed=BASE_SEED + 50_000 + scenario_index)
    samc_tuning = tune_samc_for_scenario(scenario.problem, seed=BASE_SEED + 90_000 + scenario_index)

    if verbose:
        print(
            "MCMC tuning:",
            {
                "beta_used": round(mcmc_tuning["beta_used"], 6),
                "sigma_t": round(mcmc_tuning["sigma_t"], 6),
                "q_hat_beta_hat": round(mcmc_tuning["q_hat_beta_hat"], 6),
                "tuning_evals": mcmc_tuning["tuning_evals"],
            },
        )
        print(
            "SAMC tuning:",
            {
                "lambda_min": round(samc_tuning["lambda_min"], 6),
                "tuning_evals": samc_tuning["tuning_evals"],
            },
        )

    records: list[dict] = []
    for budget in BUDGETS:
        if verbose:
            print(f"  Budget {budget:,}: running {N_REPEATS} replicates...")
        for rep in range(N_REPEATS):
            records.extend(
                run_single_budget_rep(
                    scenario,
                    budget=budget,
                    rep=rep,
                    scenario_index=scenario_index,
                    mcmc_tuning=mcmc_tuning,
                    samc_tuning=samc_tuning,
                )
            )

    metadata = {
        "scenario": scenario,
        "mcmc_tuning": mcmc_tuning,
        "samc_tuning": samc_tuning,
    }
    return records, metadata


def unique_sorted(records: list[dict], key: str) -> list:
    return sorted({row[key] for row in records})


def filter_rows(records: list[dict], **kwargs) -> list[dict]:
    out = []
    for r in records:
        keep = True
        for k, v in kwargs.items():
            if r[k] != v:
                keep = False
                break
        if keep:
            out.append(r)
    return out


def summarize_records(records: list[dict]) -> list[dict]:
    rows = []
    for scenario in unique_sorted(records, "scenario"):
        for budget in unique_sorted(records, "budget"):
            subset_sb = filter_rows(records, scenario=scenario, budget=budget)
            if not subset_sb:
                continue
            exact_p = float(subset_sb[0]["exact_p"])
            for method in unique_sorted(subset_sb, "method"):
                s = [r for r in subset_sb if r["method"] == method]
                est = np.asarray([r["estimate"] for r in s], dtype=float)
                se2 = np.asarray([r["squared_error"] for r in s], dtype=float)
                logerr = np.asarray([r["abs_log10_error"] for r in s], dtype=float)
                var_hat = np.asarray([r["variance_estimate"] for r in s], dtype=float)
                eval_excl = np.asarray([r["eval_excl_tuning"] for r in s], dtype=float)
                eval_incl = np.asarray([r["eval_incl_tuning"] for r in s], dtype=float)
                t_excl = np.asarray([r["wall_time_sec"] for r in s], dtype=float)
                t_incl = np.asarray([r["wall_time_incl_tuning_sec"] for r in s], dtype=float)

                emp_var = float(np.var(est, ddof=1)) if est.size > 1 else np.nan
                finite_var_hat = var_hat[np.isfinite(var_hat)]
                mean_var_hat = float(np.mean(finite_var_hat)) if finite_var_hat.size else np.nan
                var_calib = float(emp_var / mean_var_hat) if (np.isfinite(emp_var) and np.isfinite(mean_var_hat) and mean_var_hat > 0) else np.nan

                rows.append({
                    "scenario": scenario,
                    "budget": int(budget),
                    "method": method,
                    "n_rep": int(est.size),
                    "exact_p": exact_p,
                    "mean_estimate": float(np.mean(est)),
                    "median_estimate": float(np.median(est)),
                    "bias": float(np.mean(est) - exact_p),
                    "rel_bias": float((np.mean(est) - exact_p) / exact_p),
                    "rmse": float(np.sqrt(np.mean(se2))),
                    "mean_abs_log10_error": float(np.mean(logerr)),
                    "empirical_var": emp_var,
                    "mean_variance_estimate": mean_var_hat,
                    "var_calibration_ratio": var_calib,
                    "mean_eval_excl_tuning": float(np.mean(eval_excl)),
                    "mean_eval_incl_tuning": float(np.mean(eval_incl)),
                    "mean_time_excl_tuning_sec": float(np.mean(t_excl)),
                    "mean_time_incl_tuning_sec": float(np.mean(t_incl)),
                    "mean_zero_rate": float(np.mean([r["zero_hits"] for r in s if np.isfinite(r["zero_hits"])])) if any(np.isfinite(r["zero_hits"]) for r in s) else np.nan,
                    "mean_q_tilt_tail_share": float(np.mean([r["q_tilt_tail_share"] for r in s if np.isfinite(r["q_tilt_tail_share"])])) if any(np.isfinite(r["q_tilt_tail_share"]) for r in s) else np.nan,
                    "mean_ess": float(np.mean([r["ess"] for r in s if np.isfinite(r["ess"])])) if any(np.isfinite(r["ess"]) for r in s) else np.nan,
                    "mean_acceptance_rate": float(np.mean([r["acceptance_rate"] for r in s if np.isfinite(r["acceptance_rate"])])) if any(np.isfinite(r["acceptance_rate"]) for r in s) else np.nan,
                    "mean_samc_max_rel_freq_error": float(np.mean([r["samc_max_rel_freq_error"] for r in s if np.isfinite(r["samc_max_rel_freq_error"])])) if any(np.isfinite(r["samc_max_rel_freq_error"]) for r in s) else np.nan,
                    "mean_samc_convergence_rate": float(np.mean([r["samc_converged"] for r in s if np.isfinite(r["samc_converged"])])) if any(np.isfinite(r["samc_converged"]) for r in s) else np.nan,
                })
    return rows


def winner_table(summary_rows: list[dict], metric: str) -> list[dict]:
    out = []
    scenarios = sorted({r["scenario"] for r in summary_rows})
    for scenario in scenarios:
        budgets = sorted({r["budget"] for r in summary_rows if r["scenario"] == scenario})
        for budget in budgets:
            rows_sb = [r for r in summary_rows if r["scenario"] == scenario and r["budget"] == budget]
            rows_sb = [r for r in rows_sb if np.isfinite(r[metric])]
            if not rows_sb:
                continue
            best = min(rows_sb, key=lambda r: r[metric])
            out.append({
                "scenario": scenario,
                "budget": budget,
                "metric": metric,
                "winner_method": best["method"],
                "winner_value": float(best[metric]),
            })
    return out


## Run Study

In [ ]:
all_scenarios = build_scenarios()
scenarios = [all_scenarios[name] for name in SELECTED_SCENARIOS]

print("Scenario overview:")
for sc in scenarios:
    print({
        "name": sc.name,
        "display": sc.display_name,
        "exact_p": sc.exact_p,
        "tail_hits": sc.exact_tail_hits,
        "n_perm": sc.exact_n_perm,
        "exact_method": sc.exact_method,
    })

all_records: list[dict] = []
scenario_metadata: dict[str, dict] = {}

study_t0 = time.perf_counter()
for i, sc in enumerate(scenarios):
    recs, meta = run_scenario_experiment(sc, scenario_index=i, verbose=True)
    all_records.extend(recs)
    scenario_metadata[sc.name] = {
        "mcmc_tuning": meta["mcmc_tuning"],
        "samc_tuning": {
            k: (v.tolist() if isinstance(v, np.ndarray) else v)
            for k, v in meta["samc_tuning"].items()
        },
        "exact_p": sc.exact_p,
    }

study_dt = time.perf_counter() - study_t0
print(f"\nCompleted study in {study_dt/60.0:.2f} minutes.")
print(f"Total method runs: {len(all_records)}")


## Summaries

In [ ]:
summary_rows = summarize_records(all_records)

print("First 12 summary rows:")
for row in summary_rows[:12]:
    print(row)

print("\nWinners by RMSE:")
for row in winner_table(summary_rows, metric="rmse"):
    print(row)

print("\nWinners by mean_abs_log10_error:")
for row in winner_table(summary_rows, metric="mean_abs_log10_error"):
    print(row)


## Efficiency Plots

RMSE is shown against average evaluation count and average wall-clock time.
Both **excluding** and **including tuning cost** are displayed.


In [ ]:
METHOD_ORDER = ["iid", "mcmc_is", "samc"]
METHOD_LABEL = {
    "iid": "IID",
    "mcmc_is": "MCMC-IS",
    "samc": "SAMC",
}
METHOD_COLOR = {
    "iid": "tab:blue",
    "mcmc_is": "tab:orange",
    "samc": "tab:green",
}


def plot_efficiency_for_scenario(summary_rows: list[dict], scenario: str) -> None:
    rows_s = [r for r in summary_rows if r["scenario"] == scenario]
    budgets = sorted({r["budget"] for r in rows_s})

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    for method in METHOD_ORDER:
        rows_m = [r for r in rows_s if r["method"] == method]
        if not rows_m:
            continue
        rows_m = sorted(rows_m, key=lambda r: r["budget"])

        x_eval_ex = np.asarray([r["mean_eval_excl_tuning"] for r in rows_m], dtype=float)
        x_eval_in = np.asarray([r["mean_eval_incl_tuning"] for r in rows_m], dtype=float)
        x_time_ex = np.asarray([r["mean_time_excl_tuning_sec"] for r in rows_m], dtype=float)
        x_time_in = np.asarray([r["mean_time_incl_tuning_sec"] for r in rows_m], dtype=float)
        y_rmse = np.asarray([r["rmse"] for r in rows_m], dtype=float)

        color = METHOD_COLOR[method]
        label = METHOD_LABEL[method]

        axes[0].plot(x_eval_ex, y_rmse, marker="o", color=color, linestyle="-", label=f"{label} (excl tune)")
        axes[0].plot(x_eval_in, y_rmse, marker="x", color=color, linestyle="--", label=f"{label} (incl tune)")

        axes[1].plot(x_time_ex, y_rmse, marker="o", color=color, linestyle="-", label=f"{label} (excl tune)")
        axes[1].plot(x_time_in, y_rmse, marker="x", color=color, linestyle="--", label=f"{label} (incl tune)")

    axes[0].set_xscale("log")
    axes[0].set_yscale("log")
    axes[0].set_xlabel("Mean evaluations")
    axes[0].set_ylabel("RMSE")
    axes[0].set_title("RMSE vs evaluations")
    axes[0].legend(fontsize=8)

    axes[1].set_xscale("log")
    axes[1].set_yscale("log")
    axes[1].set_xlabel("Mean wall time (sec)")
    axes[1].set_ylabel("RMSE")
    axes[1].set_title("RMSE vs wall time")
    axes[1].legend(fontsize=8)

    fig.suptitle(f"Efficiency: {scenario}")
    plt.tight_layout()
    plt.show()


for sc in SELECTED_SCENARIOS:
    plot_efficiency_for_scenario(summary_rows, sc)


## Boxplots at Largest Budget

Shows distribution of estimates and reported variance estimates across replications.


In [ ]:
def boxplots_for_scenario(records: list[dict], scenario: str, budget: int) -> None:
    rows = [r for r in records if r["scenario"] == scenario and r["budget"] == budget]
    if not rows:
        return
    exact_p = float(rows[0]["exact_p"])

    est_data = []
    var_data = []
    labels = []
    for method in METHOD_ORDER:
        rm = [r for r in rows if r["method"] == method]
        if not rm:
            continue
        est = np.asarray([r["estimate"] for r in rm], dtype=float)
        var = np.asarray([r["variance_estimate"] for r in rm], dtype=float)
        var = var[np.isfinite(var) & (var > 0.0)]
        if var.size == 0:
            var = np.asarray([np.nan], dtype=float)
        est_data.append(np.log10(np.maximum(est, 1e-300)))
        var_data.append(np.log10(var))
        labels.append(METHOD_LABEL[method])

    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.5))
    axes[0].boxplot(est_data, tick_tick_labels=labels, showfliers=False)
    axes[0].axhline(np.log10(exact_p), linestyle="--", color="black", linewidth=1.2, label="log10 exact")
    axes[0].set_title("Estimator distribution")
    axes[0].set_ylabel("log10(estimate)")
    axes[0].legend()

    axes[1].boxplot(var_data, tick_tick_labels=labels, showfliers=False)
    axes[1].set_title("Reported variance estimates")
    axes[1].set_ylabel("log10(variance estimate)")

    fig.suptitle(f"{scenario} @ budget={budget:,}")
    plt.tight_layout()
    plt.show()


for sc in SELECTED_SCENARIOS:
    b = max(BUDGETS)
    boxplots_for_scenario(all_records, sc, budget=b)


## Method-Specific Diagnostics

- MCMC-IS: tilted-tail occupancy `q`, ESS, acceptance
- SAMC: max relative frequency error and convergence rate
- IID: zero-hit rate


In [ ]:
def diagnostics_for_scenario(records: list[dict], scenario: str) -> None:
    budgets = [max(BUDGETS)] if LARGEST_BUDGET_ONLY_DIAGNOSTICS else sorted({r["budget"] for r in records if r["scenario"] == scenario})

    for budget in budgets:
        rows = [r for r in records if r["scenario"] == scenario and r["budget"] == budget]
        if not rows:
            continue

        mcmc_rows = [r for r in rows if r["method"] == "mcmc_is"]
        samc_rows = [r for r in rows if r["method"] == "samc"]
        iid_rows = [r for r in rows if r["method"] == "iid"]

        print(f"\nDiagnostics: {scenario}, budget={budget:,}")
        if mcmc_rows:
            q = np.asarray([r["q_tilt_tail_share"] for r in mcmc_rows], dtype=float)
            ess = np.asarray([r["ess"] for r in mcmc_rows], dtype=float)
            acc = np.asarray([r["acceptance_rate"] for r in mcmc_rows], dtype=float)
            print({
                "mcmc_q_mean": float(np.mean(q)),
                "mcmc_q_median": float(np.median(q)),
                "mcmc_ess_mean": float(np.mean(ess)),
                "mcmc_acc_mean": float(np.mean(acc)),
            })

        if samc_rows:
            ferr = np.asarray([r["samc_max_rel_freq_error"] for r in samc_rows], dtype=float)
            conv = np.asarray([r["samc_converged"] for r in samc_rows], dtype=float)
            print({
                "samc_max_rel_freq_error_mean": float(np.mean(ferr)),
                "samc_convergence_rate": float(np.mean(conv)),
            })

        if iid_rows:
            zero = np.asarray([r["zero_hits"] for r in iid_rows], dtype=float)
            print({"iid_zero_hit_rate": float(np.mean(zero))})

        fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))

        if mcmc_rows:
            axes[0].boxplot([
                np.asarray([r["q_tilt_tail_share"] for r in mcmc_rows], dtype=float),
                np.asarray([r["ess"] for r in mcmc_rows], dtype=float),
                np.asarray([r["acceptance_rate"] for r in mcmc_rows], dtype=float),
            ], labels=["q", "ESS", "acc"])
            axes[0].set_title("MCMC-IS diagnostics")
        else:
            axes[0].axis("off")

        if samc_rows:
            axes[1].boxplot([
                np.asarray([r["samc_max_rel_freq_error"] for r in samc_rows], dtype=float),
                np.asarray([r["samc_pi0"] for r in samc_rows], dtype=float),
            ], labels=["max rel freq err", "pi0"])
            axes[1].set_title("SAMC diagnostics")
        else:
            axes[1].axis("off")

        if iid_rows:
            axes[2].hist(np.asarray([r["tail_hits"] for r in iid_rows], dtype=float), bins=min(15, len(iid_rows)), alpha=0.8)
            axes[2].set_title("IID tail hits")
            axes[2].set_xlabel("tail hits")
        else:
            axes[2].axis("off")

        fig.suptitle(f"{scenario} diagnostics @ budget={budget:,}")
        plt.tight_layout()
        plt.show()


for sc in SELECTED_SCENARIOS:
    diagnostics_for_scenario(all_records, sc)


## Tuning Metadata Snapshot

Useful for the Methods section in your article.


In [ ]:
print(json.dumps(scenario_metadata, indent=2))